In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource']


# DB-Specific

In [3]:
from lib import beatport
mio   = beatport.MusicDBIO(verbose=False)
webio = beatport.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant Beatport DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Beatport


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Artists:   {0}".format(len(localArtists.get())))
print("   Master Artists:  {0}".format(len(masterArtists.get())))
print("   Errors:          {0}".format(len(errors.get())))
print("   Search Artists:  {0}".format(searchArtists.shape[0]))
print("   Known IDs:       {0}".format(len(knownArtists)))

Beatport Search Results
   Local Artists:   116776
   Master Artists:  0
   Errors:          29
   Search Artists:  203405
   Known IDs:       88163


# Download Artist Data

In [6]:
mio   = beatport.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = beatport.RawWebData(debug=False)

In [32]:
useSearchData = True

if useSearchData is True:
    artistNames      = searchArtists.sort_values(by="Num", ascending=False)
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())].rename(columns={"Ref": "URL"})

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(localArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  39998
#   Artist Names To Get:  33523

Beatport Search Results
   Available Names:      203405
   Known Artist Names:   185226
   Artist Names To Get:  18173


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "10:00am")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["Name"]
    artistURL  = row["URL"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistURL=artistURL)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(4.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting Beatport ArtistIDs] Start    ==> Time Is 2022-04-25 19:55:17
========================= termTime(day=tomorrow,time=6:50am) =========================
   ====> Terminate Time Set To 2022-04-26 06:50:00 <====
   ====> Will Terminate Process 10 Hours and 54 Minutes From Now
Getting Data For FP (12115)                                        True
Getting Data For Rob Vegas (320165)                                True
Getting Data For Goodtimes (722660)                                True
Getting Data For Randi Soyland (320127)                            True
Getting Data For base2 (722800)                                    True
Getting Data For Khrizz Arias (121095)                             True
Getting Data For DJ Dayvi (230375)                                 True
Getting Data For Paul Sureno (121097)                              True
Getting Data For Alex Mart (722922)                                True
Getting Data For Emrah Ozdemir (121115)                          

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For American Intrigue (72332)                         True
Getting Data For Record Playerz (723690)                           True
Getting Data For Bronster Bridge (723751)                          True
Getting Data For Footfull (319856)                                 True
Getting Data For WickedestSound (723747)                           True
Getting Data For T.Garcia (723740)                                 True
Getting Data For Dieter Meier (120841)                             True
Getting Data For PACCI (723737)                                    True
Getting Data For Al Agami (120849)                                 True
Getting Data For Loop Corporation (120855)                         True
Getting Data For Meltzer (723765)                                  True
Getting Data For K21 (120857)                                      True
Getting Data For Kiki Kudo (723670)                                True
Getting Data For SFA's House (723664)                           

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For Deykin (723918)                                   True
Getting Data For Zeigenbock Kopf (31981)                           True
Getting Data For Edgar Storm (723902)                              True
Getting Data For Pluton Kids (230511)                              True
Getting Data For Anthonio Napoli (72389)                           True
Getting Data For Nathan Michel (31984)                             True
Getting Data For Scorsayzee (723858)                               True
Getting Data For Original Hamster (31983)                          True
Getting Data For Meddie Mercury (723838)                           True
Getting Data For Stephenie Coker (72382)                           True
Getting Data For Wighnomy Bros (230486)                            True
Getting Data For Sexlife (319839)                                  True
Getting Data For Fox N Feel (120835)                               True
Getting Data For Nicolé Mhist (723612)                          

...

Getting Data For Kingseyes (320216)                                True
Getting Data For Agez (722350)                                     True
Getting Data For Harlem Zip Code (3202)                            True
Getting Data For Frida Harnesk (121225)                            True
Getting Data For Patrick Daniels (320278)                          True
Getting Data For Outerlands (722585)                               True
Getting Data For Emiliano Ferreyra (121187)                        True
Getting Data For The Lazy Fingers (320178)                         True
Getting Data For Nikki Bliss (230320)                              True
Getting Data For Dub Majesty (722609)                              True
Getting Data For C'buda M (722606)                                 True
Getting Data For meus (722604)                                     True
Getting Data For Moon Williams (722596)                            True
Getting Data For Oskalizator (722584)                           

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For Uncommenn (722141)                                True
Getting Data For THE TRIPPIN' FEAR (722188)                        True
Getting Data For Glitch Hiker (722178)                             True
Getting Data For Idris Muhammad (121323)                           True
Getting Data For Juan Chousa & Kanike (722162)                     True
Getting Data For Magnit (320255)                                   True
Getting Data For A.J. Leo (722154)                                 True
Getting Data For Diamond Boy Luis (12134)                          True
Getting Data For Villey (722120)                                   True
Getting Data For Billow Jazz (722323)                              True
Getting Data For Sai Khuntia (722110)                              True
Getting Data For Fon-Kin (7221)                                    True
Getting Data For ZAN (121359)                                      True
Getting Data For Gary Optim (72209)                             

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For Damian Wild (12127)                               True
Getting Data For Dom & Roland (121272)                             True
Getting Data For Mallow (722261)                                   True
Getting Data For Tuner (320226)                                    True
Getting Data For Matt The Bot (722256)                             True
Getting Data For Awesome Dawson (121275)                           True
Getting Data For The Untold (722245)                               True
Getting Data For weish (722243)                                    

In [ ]:
localArtists.save(data=localArtistsDict)

In [ ]:
from lib.beatport import moveLocalFiles
moveLocalFiles()

In [ ]:
from utils import PoolIO
pio = PoolIO("Beatport")
pio.parse()
pio.merge()
pio.meta()
pio.sum()
pio.search()

# Create Starter

In [ ]:
from collections import Counter
from ioutils import FileIO, HTMLIO
io = FileIO()
hio = HTMLIO()
cntr = Counter()
refs = []
for modVal in range(100):
    for ifile in mio.dir.getRawModValDataDir(modVal).glob("*.p"):
        bsdata = hio.get(io.get(ifile))
        refs  += [(ref.attrs['data-artist'],ref.attrs['href'],ref.text) for ref in bsdata.findAll('a') if ref.attrs.get('data-artist') is not None]
    if (modVal+1) % 5 == 0:
        print(modVal,'\t',len(refs))

In [ ]:
df = DataFrame(refs)
N  = df[0].value_counts()
N.name = "Num"
df = df.drop_duplicates()
df.index = df[0]
df.index.name = None
df = df.drop([0], axis=1)
df.columns = ["Ref", "Name"]
df = df.join(N)
df.sort_values(by='Num', ascending=False)

In [ ]:
mio.data.saveSearchArtistData(data=df)